In [ ]:
import os
import cv2
import pickle
import mediapipe as mp
from screeninfo import get_monitors

In [ ]:
def bounding_box(width_frame, height_frame, x_landmarks, y_landmarks):
    xmin, xmax = min(x_landmarks), max(x_landmarks)
    ymin, ymax = min(y_landmarks), max(y_landmarks)
    width, height = xmax - xmin, ymax - ymin
    if width > height:
        delta_x = 0.1 * width
        delta_y = delta_x + (width - height) / 2
    else:
        delta_y = 0.1 * height
        delta_x = delta_y + (height - width) / 2
    start_x = max(int((xmin - delta_x) * width_frame), 0)
    start_y = max(int((ymin - delta_y) * height_frame), 0)
    end_x   = min(int((xmax + delta_x) * width_frame), width_frame)
    end_y   = min(int((ymax + delta_y) * height_frame), height_frame)
    landmarks_norm = []
    for i in range(len(x_landmarks)):
        x = x_landmarks[i]
        y = y_landmarks[i]
        cx, cy = int(x * width_frame), int(y * height_frame)
        x_norm = (cx - start_x) / (end_x - start_x)
        y_norm = (cy - start_y) / (end_y - start_y)
        landmarks_norm.append([x_norm, y_norm])

    result = {'landmarks': landmarks_norm,
              'bounding_box_start': (start_x, start_y),
              'bounding_box_end': (end_x, end_y)}
    return result

In [ ]:
"""
Mô phỏng qua camera

Nếu tay trái => tay phải
Nếu tay phải => tay phải

tay trái ngoài thật là tay trái khi nhận diện, ngược lại
"""

mp_hands = mp.solutions.hands
mp_pose = mp.solutions.pose

mp_drawing = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)

monitor = get_monitors()[0]
screen_w = monitor.width
screen_h = monitor.height
cap.set(cv2.CAP_PROP_FRAME_WIDTH, screen_w)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, screen_h)

DATA = []
LABELS = []

# with (mp_hands.Hands(
#     static_image_mode=False,
#     max_num_hands=2,
#     min_detection_confidence=0.5,
#     min_tracking_confidence=0.5) as hands), (mp_pose.Pose(
#     static_image_mode=False,
#     model_complexity=1,
#     smooth_landmarks=True,
#     min_detection_confidence=0.5,
#     min_tracking_confidence=0.5
#     ) as pose):
with mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=2,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        ) as hands, mp_pose.Pose(
            static_image_mode=False,
            model_complexity=1,
            smooth_landmarks=True,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        ) as pose:

    while True:
        x_ = []
        y_ = []
        ret, frame = cap.read()
        frame = cv2.flip(frame, 1)
        # frame = cv2.resize(frame, (screen_w, screen_h))
        H, W, _ = frame.shape
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        hand_results = hands.process(frame_rgb)
        pose_results = pose.process(frame_rgb)

        if hand_results.multi_hand_landmarks and hand_results.multi_handedness:
            for hand_landmarks in hand_results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(
                    frame,
                    hand_landmarks,
                    mp_hands.HAND_CONNECTIONS,
                    mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=3),
                    mp_drawing.DrawingSpec(color=(255, 0, 0), thickness=2)
                )
        
        if pose_results.pose_landmarks:
            mp_drawing.draw_landmarks(
                frame,
                pose_results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS
            )

            # for hand_landmarks, handedness in zip(hand_results.multi_hand_landmarks,hand_results.multi_handedness):
            #     label = handedness.classification[0].label  # "Left" hoặc "Right"

            #     x_landmarks = []
            #     y_landmarks = []
            #     hand_data = []
            #     if label == "Left":
            #         for lm in hand_landmarks.landmark:
            #             x_landmarks.append(1- lm.x)
            #             y_landmarks.append(lm.y)
            #             hand_data.append([1- lm.x, lm.y])
            #     else:
            #         for lm in hand_landmarks.landmark:
            #             x_landmarks.append(lm.x)
            #             y_landmarks.append(lm.y)
            #             hand_data.append([lm.x, lm.y])
                # if len(x_landmarks) + len(y_landmarks) == 42:
                #     output_bounding_box = bounding_box(W, H, x_landmarks, y_landmarks)
                #     landmarks_norm = output_bounding_box['landmarks']
                #     DATA.append(landmarks_norm)
                #     (start_x, start_y) = output_bounding_box['bounding_box_start']
                #     (end_x, end_y) = output_bounding_box['bounding_box_end']

                    # cv2.rectangle(frame, (start_x, start_y), (end_x, end_y), (0, 255, 0), 2)
                    # cv2.putText(frame, label, (start_x, start_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        cv2.imshow("Hands with Bounding Box", frame)

        if cv2.waitKey(1) & 0xFF == 27:
            break

cap.release()
cv2.destroyAllWindows()

#### YOLO

In [1]:
from ultralytics import YOLO

model = YOLO("yolo11n-pose.pt")

results = model(frame)

OpenCV bindings requires "numpy" package.
Install it via command:
    pip install numpy


ModuleNotFoundError: No module named 'numpy.core.multiarray'